# 20 - Reading the referee

**Purpose.** To explain what notebook `19` established about PixInsight, and what a reader should
now believe when the two tools are put side by side. `19` ran the harness and made the numbers,
and is written for someone *checking* the work. This one is written for someone *deciding what to
do next* - which after this session means deciding what the engine half of build step 5 can safely
assume.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `pi_contract_constants.json` and `pi_contract.csv`, and `bias_sweep.csv` where the
quantisation story needs 77 gains rather than two frames. If any of it disagreed with `results/`,
`results/` would be right and this notebook would be the bug.

**The shape of this session, stated before any number.** Contract 1 is the cheapest of the three
and the one everything else rests on: it does not compare a *result*, it establishes that the two
tools are looking at the same pixels and speaking the same units. All of that passed, exactly and
not approximately. Then it compared the one thing it was built to compare - noise estimators - and
the inherited prediction about them was wrong, in a direction and for a reason worth knowing.

**Three traps were checked rather than believed, and none of them announces itself.** Each would
have produced numbers that looked entirely reasonable. Section 2 puts a figure on what each would
have cost.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
read = lambda n: json.loads((RESULTS / n).read_text())

K = read("pi_contract_constants.json")      # notebook 19, this session
contract = pd.read_csv(RESULTS / "pi_contract.csv")
sweep = pd.read_csv(RESULTS / "bias_sweep.csv")

PLANES = ["R", "G1", "G2", "B"]
ref = K["referee"]["value"]

print(f"notebook 19 published {len(K)} constants on {K['referee']['measured_on']}, "
      f"from {K['referee']['source_frames']} frames")
print(f"referee: {ref['application']} {ref['version']} build {ref['build']}")
print(f"  {ref['command']}")
print()
print("the contract table:")
print(f"  {len(contract)} rows -- two frames x four CFA planes")
print(f"  {len(contract.columns)} columns -- both tools' five statistics, both MADs, "
      f"three noise estimators")

## 1. What "the same pixels" means, and why it needed proving

A referee is only useful if it is refereeing the same match. Three things had to be true before
any comparison of noise estimates could mean anything, and none of them is obvious from the
outside.

The two tools had to **open the same array**. PixInsight could reasonably have debayered a CFA
frame on load - most software does - and had it done so, every variance below would have been
measuring an interpolation kernel. It does not: it opens the mosaic as a single-channel mono
image, 3840 x 2160, one channel.

They had to **cut it into the same planes**. Both split a Bayer mosaic four ways, and both call
the results by index rather than by colour, so agreeing on the count is not agreeing on the
mapping.

And they had to **mean the same thing by a number**. PixInsight works in [0, 1]; this project
works in ADC counts. One conversion stands between them and it is not the obvious one.

In [ ]:
sa = K["split_agreement"]["value"]
print("contract 1's precondition, measured:")
print(f"  largest disagreement in a plane MINIMUM : {sa['max_abs_min_diff']} ADC counts")
print(f"  largest disagreement in a plane MAD     : {sa['max_abs_mad_diff']:.2e} ADC counts")
print(f"  largest disagreement in a plane STD     : {sa['max_abs_std_diff_ppm']:.2e} ppm")
print()
print("Not 'close'.  The minima are identical: both are one pixel's value and no arithmetic")
print("has touched either.  The MAD and std disagreements are float round-off in the")
print("conversion, six to seven orders of magnitude below one ADC count.")

## 2. The three traps, and what each would have cost

### Trap 1 - the divisor

PixInsight normalises by the **container maximum 65535**. Not by full scale 4095, and not by the
sensor's saturation level 65520. Composed with this project's unit, the factor is **4095.9375**.

Using 4095 is not a wild error. It is 0.023% - two parts in ten thousand - which is smaller than
most things anyone would bother checking, and it would have applied silently to every number
crossing the boundary in either direction, forever.

In [ ]:
scale = K["unit_scale"]["value"]
wrong = 4095.0
print(f"correct: 65535/16 = {scale}")
print(f"wrong  : {wrong}")
print(f"error  : {1e2*(wrong/scale - 1):.4f}%  ({1e6*(1 - wrong/scale):.0f} ppm low)")
print()
print("what that does to numbers this project actually publishes:")
for label, counts in [("pedestal at gain 0, offset 10", 10.0),
                      ("read noise at gain 200", 1.077),
                      ("linear_to_at_least at gain 50", 3958.0)]:
    print(f"  {label:32s} {counts:8.3f} -> {counts*wrong/scale:8.3f} counts  "
          f"(off by {counts*(1 - wrong/scale):.4f})")
print()
print("Small enough to pass for rounding in every single case.  That is the whole argument for")
print("measuring it: an error that never looks like an error is never found by looking.")

### Trap 2 - the plane order, and the statistic that cannot see it

`SplitCFA` enumerates the 2x2 tile in PixInsight's `(x, y)` order with `y` varying fastest, and
PixInsight's first coordinate is the column. Translated into numpy's `(row, col)`, **the two
greens come back transposed**. R and B sit on the diagonal of the tile, so transposing leaves them
exactly where they were - which is what makes this survive a casual check.

The claim came with an instruction: do not compare medians. Notebook `19` did it both ways.

In [ ]:
order = K["cfa_order"]["value"]
naive = ["R", "G1", "G2", "B"]
print(f"PixInsight's order : {order}")
print(f"naive RGGB reading : {naive}")
print(f"                     {'  '.join('^^' if a != b else '  ' for a, b in zip(order, naive))}")
print()
g = contract[contract.plane.isin(["G1", "G2"])]
print("the two greens, as PixInsight reports them:")
print(g[["frame", "cfa", "plane", "pi_min", "pi_median", "pi_mrs"]].to_string(
    index=False, float_format=lambda v: f"{v:9.4f}"))
print()
for frame in contract.frame.unique():
    sub = g[g.frame == frame]
    mins = sub.pi_min.tolist()
    meds = sub.pi_median.tolist()
    print(f"{frame:6s}: minima differ by {abs(mins[0]-mins[1]):.1f} counts; "
          f"medians differ by {abs(meds[0]-meds[1]):.1f}")
print()
print("So a median comparison passes the WRONG mapping on both frames.  The two greens share a")
print("median on any real frame -- they are the same colour on the same sensor -- and the swap")
print("is invisible to it.  The minimum is one pixel, and one pixel cannot be in two places.")

### Trap 3 - whose variance

`pcl::Variance` returns the sample variance, dividing by n-1. `ndarray.std()` returns the
population one, dividing by n. This one is different in kind from the other two: on the frames
this project works with, **it does not matter at all**.

In [ ]:
n = 2073600                                  # one plane of a full frame
print(f"a full-frame plane is {n:,} px")
print(f"  sqrt(n/(n-1)) - 1 = {np.sqrt(n/(n-1)) - 1:.2e}  ->  {1e6*(np.sqrt(n/(n-1))-1):.2f} ppm")
for small in (12, 100, 1000):
    print(f"  at n = {small:5d}: {1e2*(np.sqrt(small/(small-1))-1):6.2f} %")
print()
print("Which is the point, and the reason it is a library constant rather than a note:")
print("it is unmeasurable where the frames are, and 4.4% where a test can check it.  Carried")
print("in pixinsight.matching_stats, proved at n=12 in tests/test_pixinsight.py.")

## 3. The estimators, and a prediction that did not survive

Everything above is plumbing. This is the comparison contract 1 exists to make.

Four numbers per plane. **MRS** is PixInsight's multiresolution support estimate, which models the
frame's structure and measures what is left. **k-sigma** is iterative clipping about the mean.
**Ours** is `1.4826 x MAD`, the estimator the frame index and every bench session use. And PI's
**plain standard deviation**, which rejects nothing and is the ceiling.

The retired project predicted `MRS < ours < ksigma` - ours sitting between PixInsight's two,
because its multiresolution rejects harder than our clip. There are no frames left behind that
claim, so it was a prediction and nothing more.

In [ ]:
no = K["noise_ordering"]["value"]
print(f"predicted: {no['predicted']}")
print(f"measured : {no['measured']}")
print(f"verdict  : {no['verdict'].upper()}")
print()
cols = ["frame", "plane", "pi_mrs", "pi_ksigma", "ours_mad_sigma", "pi_std"]
print(contract[cols].to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print()
print(f"ours / MRS runs {no['ours_over_mrs']['min']:.2f} to {no['ours_over_mrs']['max']:.2f} "
      f"across all eight planes.")
print("PixInsight's two land within 10% of each other.  Ours sits half again as high, on every")
print("plane of both frames.  It is not between them anywhere.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, frame in zip(axes, ["bias", "light"]):
    sub = contract[contract.frame == frame].set_index("plane").loc[PLANES]
    x = np.arange(len(PLANES))
    ax.plot(x, sub.pi_mrs, "o-", label="PI MRS")
    ax.plot(x, sub.pi_ksigma, "s-", label="PI k-sigma")
    ax.plot(x, sub.ours_mad_sigma, "^-", label="ours, 1.4826 x MAD")
    ax.set_xticks(x); ax.set_xticklabels(PLANES)
    ax.set_title(f"{frame}  (PI std {sub.pi_std.min():.2f} to {sub.pi_std.max():.2f})")
    ax.set_ylabel("ADC counts"); ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
axes[0].legend(loc="lower left", fontsize=7)
fig.suptitle("Three estimators on the same planes -- ours is above both, everywhere", y=1.02)
fig.tight_layout()
plt.show()

### Why it is not a rejection difference

Two things could put our number above PixInsight's. We could be computing the same quantity and
getting a different answer, which would be a bug. Or we could be computing a different quantity,
which is a finding.

PixInsight reports its own median absolute deviation, and so do we. Same definition, same pixels.

In [ ]:
print(contract[["frame", "plane", "our_mad", "pi_mad", "d_mad"]].to_string(
    index=False, float_format=lambda v: f"{v:12.9f}"))
print(f"\nlargest disagreement: {contract.d_mad.abs().max():.2e} ADC counts")
print("\nThe MADs are the same number.  So the gap is not about what gets rejected -- it is")
print("about what happens to a MAD when the data are integers and the spread is small.")

## 4. The grid under our estimator

A median absolute deviation over integer data is an integer or a half-integer. So `1.4826 x MAD`
can only land on multiples of 1.4826 ADC counts, and the grid is **absolute** - it does not shrink
when the thing being measured does.

On the bias, every plane's MAD is exactly 1. Our estimator returns 1.4826 on all four, where
PixInsight's plain standard deviation of those same pixels is about 1.09. We read roughly a third
high, and we could not have read anything else: the neighbouring values our estimator can take are
0 and 2.965.

Two frames could be a coincidence. `bias_sweep.csv` has 77 gains - 62 of them inside the project's
gain domain, and the cell below keeps only those - with read noise measured two independent ways:
`R_sd` from pair differences, which is not quantised, and `R_mad`, which is.

In [ ]:
per_gain = (sweep.sort_values("offset").groupby("gain").first().reset_index()
                 .dropna(subset=["R_mad", "R_sd"]))
per_gain = per_gain[per_gain.gain <= 450]        # the gain domain
rung = per_gain[per_gain.R_mad.round(4) == 1.0484]

print(f"R_mad returns the SAME value (1.0484) at {len(rung)} of the {len(per_gain)} gains "
      f"at or below 450:")
print(f"  gains {int(rung.gain.min())} to {int(rung.gain.max())}")
print(f"  over which the true read noise runs {rung.R_sd.min():.3f} to {rung.R_sd.max():.3f} "
      f"ADC counts")
print(f"  -- a factor of {rung.R_sd.max()/rung.R_sd.min():.2f}, called identical.")
print()
print(f"Meanwhile R_sd gives {per_gain.R_sd.round(4).nunique()} distinct values for those "
      f"{len(per_gain)} gains.  This is not an argument about which")
print("estimator is 'better' in general -- MAD is robust and that is why the index uses it.")
print("It is a statement about where it stops working: a few counts of spread on integer data.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(per_gain.gain, per_gain.R_sd, "-", lw=1.2, label="R_sd -- pair difference, unquantised")
ax.plot(per_gain.gain, per_gain.R_mad, "-", lw=1.2, label="R_mad -- MAD, on a 1.0483 grid")
ax.axvspan(rung.gain.min(), rung.gain.max(), alpha=0.12,
           label=f"one R_mad value, {len(rung)} gains")
ax.set_xlabel("gain"); ax.set_ylabel("read noise, ADC counts")
ax.set_yscale("log"); ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("The staircase: where a MAD stops resolving  (results/bias_sweep.csv)")
fig.tight_layout()
plt.show()

### What this changes, and what it does not

**It changes nothing that is published.** No read-noise constant in this repo comes from a spatial
MAD. `bias_sweep.csv` and `ptc_constants.json` publish pair-difference sigma; `cold_constants.json`
publishes the temporal spread across a bias block. Both sidestep the grid entirely, and both were
chosen that way for reasons written down at the time - this session did not rescue anything.

**It changes how to read one column.** `sigma` in `results/frame_index.csv` is `1.4826 x MAD`, and
it now has a known floor: wherever a frame's spread is a few counts, that column reads high, and
two frames with genuinely different noise can share a value. It is a *description* of a frame,
which is what the index is for, and it was never evidence about the sensor.

**It changes which referee to use.** For a spread of a few counts, PixInsight's MRS is the better
instrument, and precisely because it is not built on a median of integers.

In [ ]:
q = K["our_sigma_quantisation"]
print(f"our_sigma_quantisation = {q['value']} {q['unit']}")
print()
print("published read noise, and how each avoids the grid:")
for f, key, how in [("bias_sweep.csv", "R_sd", "pair difference / sqrt(2)"),
                    ("ptc_constants.json", "system_gain", "slope of pair-difference variance"),
                    ("cold_constants.json", "read_noise_cold",
                     "temporal std across a 20-frame bias block")]:
    print(f"  {f:24s} {key:18s} {how}")
print()
print("affected: results/frame_index.csv, column `sigma` -- a description, not evidence.")

## 5. What is unblocked, and what is still shut

Contract 1 is done. Its job was never to produce a physical constant; it was to make the other two
contracts *mean* something, and it has.

In [ ]:
print("LICENSED by contract 1")
print("  - any PixInsight number about this sensor's frames converts to ADC counts exactly")
print("  - SplitCFA output CFA0..CFA3 maps to", K["cfa_order"]["value"])
print("  - our plane statistics and PixInsight's are the same numbers")
print("  - pjsr/ has a harness that fails in bounded time and says why")
print()
print("STILL SHUT")
print("  contract 2  g and R against PixInsight's own estimate")
print("              -- needs a bias PAIR, and the 16-bit subtraction trap is UNCHECKED")
print("  contract 3  stacking noise reduction vs ImageIntegration's reported figure")
print("  the engine  registration + integration -> eta_comb, the SNR estimator's")
print("              repeatability, and MISSION's four ranked pairs")

### The one trap left in the queue

Subtracting two 16-bit unsigned images inside PixInsight **clips every negative difference**. For a
bias pair that halves the apparent read noise, and it fails quietly - the number stays entirely
plausible. The fix is to subtract in 32-bit float with a `+0.5` pedestal: `A - B + 0.5`, with
`rescale` and `truncate` both off, which moves the mean without touching the standard deviation
and keeps the distribution inside [0, 1].

It is unchecked because contract 1 subtracts nothing. It is the **first** thing contract 2 must
verify, before any number it produces is believed, and the way to verify it is to inject a known
sigma into a synthetic pair and assert recovery rather than to inspect a real one - a halved read
noise on a real bias looks exactly like a good camera.

### The shortest path to the definition of done

MISSION's gate is four ranked pairs, and session 06's frames are already on disk and balanced.
What stands between them is registration and integration, not another night. The engine is now the
whole of it, and it is the only thing that is.

In [ ]:
print("what MISSION's definition of done still needs, in order:")
print("  1. the engine: register + integrate one rung, headless, through pjsr/")
print("  2. eta_comb from a real stack -- the measured loss against ideal sqrt(N)")
print("  3. the SNR estimator's repeatability, which sets how far apart a PAIR must be")
print("  4. four ranked pairs, one straddling HCG, predicted apart and measured")
print()
print("None of the four needs a new capture session.  All four need the same piece of code.")